In [0]:
# ============================================================
# NOTEBOOK 0c — BRONZE + RAW LOAD
# O&G Rig Operations Intelligence Platform
# Reads from staging, resolves SKs via dim lookups,
# incrementally loads bronze facts and appends to raw archive
# Depends on: Notebook_0b (staging) + Notebook_0a (dims)
# ============================================================

# CELL 1 — SETUP
from pyspark.sql import functions as F

# Create schemas if not exists
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.raw")

# Timestamp format used throughout for recorded_at parsing
# TS_FMT = "yyyy-MM-dd'T'HH:mm:ss'Z'"
TS_FMT = "yyyy-MM-dd\\'T\\'HH:mm:ss\\'Z\\'"

# Helper — get watermark (max date_id) from a bronze fact table
# Returns 0 if table is empty or does not exist (triggers full load)
def get_watermark(table: str) -> int:
    try:
        result = spark.sql(f"""
            SELECT MAX(date_id) as max_id
            FROM workspace.bronze.{table}
        """).collect()[0]["max_id"]
        watermark = result if result else 0
        print(f"  → {table} watermark: {watermark}")
        return watermark
    except:
        print(f"  → {table} not found — full load")
        return 0

print("✓ Setup complete")
print("✓ Schemas: workspace.bronze, workspace.raw")
print("✓ Watermark helper ready")

✓ Setup complete
✓ Schemas: workspace.bronze, workspace.raw
✓ Watermark helper ready


In [0]:
# CELL 2 — CREATE BRONZE FACT TABLES (one-time setup)
# Uses CREATE TABLE IF NOT EXISTS — safe to re-run, never drops data
# Column order follows our standards:
#   1. Natural business ID (if exists)
#   2. date_id + recorded_at
#   3. Surrogate keys (FKs) using [entity]_id_sk naming
#   4. Natural keys for traceability (rig_id, crew_id etc.)
#   5. Numeric measures
#   6. loaded_at

spark.sql("""
    CREATE TABLE IF NOT EXISTS workspace.bronze.fact_rig_daily_ops (
        date_id                 INT,
        recorded_at             TIMESTAMP,
        rig_id_sk               INT,
        region_id_sk            INT,
        well_id_sk              INT,
        well_type_id_sk         INT,
        status_id_sk            INT,
        shift_id_sk             INT,
        downtime_reason_id_sk   INT,
        cost_category_id_sk     INT,
        rig_id                  STRING,
        drilling_hours          DOUBLE,
        downtime_hours          DOUBLE,
        npt_hours               DOUBLE,
        rop_ft_hr               DOUBLE,
        cost_per_day            DOUBLE,
        sla_met                 INT,
        loaded_at               TIMESTAMP
    ) USING DELTA
""")

spark.sql("""
    CREATE TABLE IF NOT EXISTS workspace.bronze.fact_equipment_events (
        event_id                STRING,
        date_id                 INT,
        recorded_at             TIMESTAMP,
        rig_id_sk               INT,
        region_id_sk            INT,
        equipment_id_sk         INT,
        failure_type_id_sk      INT,
        rig_id                  STRING,
        failure_flag            INT,
        downtime_caused_hrs     DOUBLE,
        sla_met                 INT,
        resolution_hrs          DOUBLE,
        loaded_at               TIMESTAMP
    ) USING DELTA
""")

spark.sql("""
    CREATE TABLE IF NOT EXISTS workspace.bronze.fact_maintenance_log (
        maintenance_id          STRING,
        date_id                 INT,
        recorded_at             TIMESTAMP,
        rig_id_sk               INT,
        region_id_sk            INT,
        equipment_id_sk         INT,
        maintenance_type_id_sk  INT,
        contractor_id_sk        INT,
        rig_id                  STRING,
        duration_hrs            DOUBLE,
        cost                    DOUBLE,
        technician_count        INT,
        loaded_at               TIMESTAMP
    ) USING DELTA
""")

spark.sql("""
    CREATE TABLE IF NOT EXISTS workspace.bronze.fact_crew_hours (
        date_id                 INT,
        recorded_at             TIMESTAMP,
        rig_id_sk               INT,
        region_id_sk            INT,
        crew_id_sk              INT,
        shift_id_sk             INT,
        rig_id                  STRING,
        crew_id                 STRING,
        hours_worked            DOUBLE,
        overtime_hours          DOUBLE,
        is_present              INT,
        npt_attributed          INT,
        loaded_at               TIMESTAMP
    ) USING DELTA
""")

spark.sql("""
    CREATE TABLE IF NOT EXISTS workspace.bronze.fact_incident_log (
        incident_id             STRING,
        date_id                 INT,
        recorded_at             TIMESTAMP,
        rig_id_sk               INT,
        region_id_sk            INT,
        crew_id_sk              INT,
        incident_type_id_sk     INT,
        equipment_id_sk         INT,
        status_id_sk            INT,
        rig_id                  STRING,
        downtime_caused_hrs     DOUBLE,
        is_recordable           INT,
        loaded_at               TIMESTAMP
    ) USING DELTA
""")

print("✓ All 5 bronze fact tables ready (CREATE IF NOT EXISTS)")

✓ All 5 bronze fact tables ready (CREATE IF NOT EXISTS)


In [0]:
# CELL 3 — DIM VALIDATION
# Check all staging values against dims before loading facts
# Warns on mismatches — does not abort (unmatched values get -1 SK)

print("Running dim validation...\n")
warnings_found = 0

def check_values(staging_table, staging_col, dim_table, dim_col, label):
    global warnings_found
    result = spark.sql(f"""
        SELECT DISTINCT s.{staging_col} as value
        FROM workspace.staging.{staging_table} s
        LEFT JOIN workspace.bronze.{dim_table} d
            ON s.{staging_col} = d.{dim_col}
            AND d.is_current = true
        WHERE s.{staging_col} IS NOT NULL
          AND d.{dim_col} IS NULL
    """).collect()
    if result:
        warnings_found += len(result)
        values = [r["value"] for r in result]
        print(f"  ⚠ {label}: {len(result)} unmatched → {values}")
    else:
        print(f"  ✓ {label}: all values matched")

# rig_id checks
check_values("stg_rig_readings",     "rig_id",          "dim_rig",              "rig_id",           "stg_rig_readings.rig_id")
check_values("stg_equipment_events", "rig_id",          "dim_rig",              "rig_id",           "stg_equipment_events.rig_id")
check_values("stg_maintenance",      "rig_id",          "dim_rig",              "rig_id",           "stg_maintenance.rig_id")
check_values("stg_crew",             "rig_id",          "dim_rig",              "rig_id",           "stg_crew.rig_id")
check_values("stg_incidents",        "rig_id",          "dim_rig",              "rig_id",           "stg_incidents.rig_id")

# equipment checks
check_values("stg_equipment_events", "equipment_type",  "dim_equipment",        "equipment_type",   "stg_equipment_events.equipment_type")
check_values("stg_maintenance",      "equipment_type",  "dim_equipment",        "equipment_type",   "stg_maintenance.equipment_type")

# status checks
check_values("stg_rig_readings",     "status",          "dim_status",           "status_name",      "stg_rig_readings.status")
check_values("stg_incidents",        "incident_status", "dim_status",           "status_name",      "stg_incidents.incident_status")

# other checks
check_values("stg_rig_readings",     "downtime_reason", "dim_downtime_reason",  "downtime_reason",  "stg_rig_readings.downtime_reason")
check_values("stg_rig_readings",     "cost_category",   "dim_cost_category",    "cost_category",    "stg_rig_readings.cost_category")
check_values("stg_rig_readings",     "well_type",       "dim_well_type",        "well_type",        "stg_rig_readings.well_type")
check_values("stg_rig_readings",     "well_name",       "dim_well",             "well_name",        "stg_rig_readings.well_name")
check_values("stg_equipment_events", "failure_type",    "dim_failure_type",     "failure_type",     "stg_equipment_events.failure_type")
check_values("stg_maintenance",      "maintenance_type","dim_maintenance_type", "maintenance_type", "stg_maintenance.maintenance_type")
check_values("stg_maintenance",      "contractor_name", "dim_contractor",       "contractor_name",  "stg_maintenance.contractor_name")
check_values("stg_incidents",        "incident_type",   "dim_incident_type",    "incident_type",    "stg_incidents.incident_type")
check_values("stg_crew",             "shift",           "dim_shift",            "shift_name",       "stg_crew.shift")

print(f"\n{'✓ No warnings — all dims validated' if warnings_found == 0 else f'⚠ {warnings_found} total unmatched values — these will resolve to -1 SK'}")

Running dim validation...

  ✓ stg_rig_readings.rig_id: all values matched
  ✓ stg_equipment_events.rig_id: all values matched
  ✓ stg_maintenance.rig_id: all values matched
  ✓ stg_crew.rig_id: all values matched
  ✓ stg_incidents.rig_id: all values matched
  ✓ stg_equipment_events.equipment_type: all values matched
  ✓ stg_maintenance.equipment_type: all values matched
  ✓ stg_rig_readings.status: all values matched
  ✓ stg_incidents.incident_status: all values matched
  ✓ stg_rig_readings.downtime_reason: all values matched
  ✓ stg_rig_readings.cost_category: all values matched
  ✓ stg_rig_readings.well_type: all values matched
  ✓ stg_rig_readings.well_name: all values matched
  ✓ stg_equipment_events.failure_type: all values matched
  ✓ stg_maintenance.maintenance_type: all values matched
  ✓ stg_maintenance.contractor_name: all values matched
  ✓ stg_incidents.incident_type: all values matched
  ✓ stg_crew.shift: all values matched

✓ No warnings — all dims validated


In [0]:
# CELL 4 — FACT_RIG_DAILY_OPS
# Grain     : rig_id + recorded_at (one reading per rig per day)
# Watermark : MAX(date_id) — only loads dates newer than last run
# Unmatched : COALESCE to -1 for all SK joins

print("Loading fact_rig_daily_ops...")
watermark = get_watermark("fact_rig_daily_ops")

spark.sql(f"""
    INSERT INTO workspace.bronze.fact_rig_daily_ops
    SELECT
        -- date columns
        CAST(DATE_FORMAT(TO_TIMESTAMP(s.recorded_at, '{TS_FMT}'), 'yyyyMMdd') AS INT)
                                                        AS date_id,
        TO_TIMESTAMP(s.recorded_at, '{TS_FMT}')         AS recorded_at,

        -- surrogate keys — COALESCE to -1 if join fails
        COALESCE(r.rig_id_sk,              -1)          AS rig_id_sk,
        COALESCE(rg.region_id_sk,          -1)          AS region_id_sk,
        COALESCE(w.well_id_sk,             -1)          AS well_id_sk,
        COALESCE(wt.well_type_id_sk,       -1)          AS well_type_id_sk,
        COALESCE(st.status_id_sk,          -1)          AS status_id_sk,
        COALESCE(sh.shift_id_sk,           -1)          AS shift_id_sk,
        COALESCE(dr.downtime_reason_id_sk, -1)          AS downtime_reason_id_sk,
        COALESCE(cc.cost_category_id_sk,   -1)          AS cost_category_id_sk,

        -- natural keys (traceability)
        s.rig_id,

        -- measures
        s.drilling_hours,
        s.downtime_hours,
        s.npt_hours,
        s.rop_ft_hr,
        s.cost_per_day,
        s.sla_met,

        -- audit
        current_timestamp()                             AS loaded_at

    FROM workspace.staging.stg_rig_readings s

    LEFT JOIN workspace.bronze.dim_rig r
        ON  s.rig_id    = r.rig_id
        AND r.is_current = true

    LEFT JOIN workspace.bronze.dim_region rg
        ON  r.region_id_sk = rg.region_id_sk
        AND rg.is_current  = true

    LEFT JOIN workspace.bronze.dim_well w
        ON  s.well_name  = w.well_name
        AND w.is_current = true

    LEFT JOIN workspace.bronze.dim_well_type wt
        ON  s.well_type   = wt.well_type
        AND wt.is_current = true

    LEFT JOIN workspace.bronze.dim_status st
        ON  s.status      = st.status_name
        AND st.is_current = true

    LEFT JOIN workspace.bronze.dim_shift sh
        ON  sh.shift_name = CASE
                WHEN HOUR(TO_TIMESTAMP(s.recorded_at, '{TS_FMT}')) >= 6
                 AND HOUR(TO_TIMESTAMP(s.recorded_at, '{TS_FMT}')) < 18
                THEN 'Day'
                ELSE 'Night'
            END
        AND sh.is_current = true

    LEFT JOIN workspace.bronze.dim_downtime_reason dr
        ON  s.downtime_reason = dr.downtime_reason
        AND dr.is_current     = true

    LEFT JOIN workspace.bronze.dim_cost_category cc
        ON  s.cost_category = cc.cost_category
        AND cc.is_current   = true

    WHERE CAST(DATE_FORMAT(TO_TIMESTAMP(s.recorded_at, '{TS_FMT}'), 'yyyyMMdd') AS INT) > {watermark}
""")

count = spark.sql("SELECT COUNT(*) as n FROM workspace.bronze.fact_rig_daily_ops").collect()[0]["n"]
if count == 0:
    raise Exception("fact_rig_daily_ops has 0 rows after load — aborting")
print(f"✓ workspace.bronze.fact_rig_daily_ops: {count:,} rows")

Loading fact_rig_daily_ops...
  → fact_rig_daily_ops watermark: 0
✓ workspace.bronze.fact_rig_daily_ops: 17,620 rows


In [0]:
# CELL 5 — FACT_EQUIPMENT_EVENTS
# Grain     : event_id
# Note      : failure_type is nullable — NULL joins correctly to -1

print("Loading fact_equipment_events...")
watermark = get_watermark("fact_equipment_events")

spark.sql(f"""
    INSERT INTO workspace.bronze.fact_equipment_events
    SELECT
        -- natural business ID
        s.event_id,

        -- date columns
        CAST(DATE_FORMAT(TO_TIMESTAMP(s.recorded_at, '{TS_FMT}'), 'yyyyMMdd') AS INT)
                                                        AS date_id,
        TO_TIMESTAMP(s.recorded_at, '{TS_FMT}')         AS recorded_at,

        -- surrogate keys
        COALESCE(r.rig_id_sk,         -1)               AS rig_id_sk,
        COALESCE(rg.region_id_sk,     -1)               AS region_id_sk,
        COALESCE(e.equipment_id_sk,   -1)               AS equipment_id_sk,
        COALESCE(ft.failure_type_id_sk,-1)              AS failure_type_id_sk,

        -- natural keys (traceability)
        s.rig_id,

        -- measures
        s.failure_flag,
        s.downtime_caused_hrs,
        s.sla_met,
        s.resolution_hrs,

        -- audit
        current_timestamp()                             AS loaded_at

    FROM workspace.staging.stg_equipment_events s

    LEFT JOIN workspace.bronze.dim_rig r
        ON  s.rig_id     = r.rig_id
        AND r.is_current = true

    LEFT JOIN workspace.bronze.dim_region rg
        ON  r.region_id_sk = rg.region_id_sk
        AND rg.is_current  = true

    LEFT JOIN workspace.bronze.dim_equipment e
        ON  s.equipment_type = e.equipment_type
        AND e.is_current     = true

    LEFT JOIN workspace.bronze.dim_failure_type ft
        ON  s.failure_type  = ft.failure_type
        AND ft.is_current   = true

    WHERE CAST(DATE_FORMAT(TO_TIMESTAMP(s.recorded_at, '{TS_FMT}'), 'yyyyMMdd') AS INT) > {watermark}
""")

count = spark.sql("SELECT COUNT(*) as n FROM workspace.bronze.fact_equipment_events").collect()[0]["n"]
if count == 0:
    raise Exception("fact_equipment_events has 0 rows after load — aborting")
print(f"✓ workspace.bronze.fact_equipment_events: {count:,} rows")

Loading fact_equipment_events...
  → fact_equipment_events watermark: 0
✓ workspace.bronze.fact_equipment_events: 88,100 rows


In [0]:
# CELL 6 — FACT_MAINTENANCE_LOG
# Grain : maintenance_id

print("Loading fact_maintenance_log...")
watermark = get_watermark("fact_maintenance_log")

spark.sql(f"""
    INSERT INTO workspace.bronze.fact_maintenance_log
    SELECT
        -- natural business ID
        s.maintenance_id,

        -- date columns
        CAST(DATE_FORMAT(TO_TIMESTAMP(s.recorded_at, '{TS_FMT}'), 'yyyyMMdd') AS INT)
                                                        AS date_id,
        TO_TIMESTAMP(s.recorded_at, '{TS_FMT}')         AS recorded_at,

        -- surrogate keys
        COALESCE(r.rig_id_sk,              -1)          AS rig_id_sk,
        COALESCE(rg.region_id_sk,          -1)          AS region_id_sk,
        COALESCE(e.equipment_id_sk,        -1)          AS equipment_id_sk,
        COALESCE(mt.maintenance_type_id_sk,-1)          AS maintenance_type_id_sk,
        COALESCE(c.contractor_id_sk,       -1)          AS contractor_id_sk,

        -- natural keys (traceability)
        s.rig_id,

        -- measures
        s.duration_hrs,
        s.cost,
        s.technician_count,

        -- audit
        current_timestamp()                             AS loaded_at

    FROM workspace.staging.stg_maintenance s

    LEFT JOIN workspace.bronze.dim_rig r
        ON  s.rig_id     = r.rig_id
        AND r.is_current = true

    LEFT JOIN workspace.bronze.dim_region rg
        ON  r.region_id_sk = rg.region_id_sk
        AND rg.is_current  = true

    LEFT JOIN workspace.bronze.dim_equipment e
        ON  s.equipment_type = e.equipment_type
        AND e.is_current     = true

    LEFT JOIN workspace.bronze.dim_maintenance_type mt
        ON  s.maintenance_type = mt.maintenance_type
        AND mt.is_current      = true

    LEFT JOIN workspace.bronze.dim_contractor c
        ON  s.contractor_name = c.contractor_name
        AND c.is_current      = true

    WHERE CAST(DATE_FORMAT(TO_TIMESTAMP(s.recorded_at, '{TS_FMT}'), 'yyyyMMdd') AS INT) > {watermark}
""")

count = spark.sql("SELECT COUNT(*) as n FROM workspace.bronze.fact_maintenance_log").collect()[0]["n"]
if count == 0:
    raise Exception("fact_maintenance_log has 0 rows after load — aborting")
print(f"✓ workspace.bronze.fact_maintenance_log: {count:,} rows")

Loading fact_maintenance_log...
  → fact_maintenance_log watermark: 0
✓ workspace.bronze.fact_maintenance_log: 8,672 rows


In [0]:
# CELL 7 — FACT_CREW_HOURS
# Grain : rig_id + crew_id + recorded_at
# Note  : two rows per rig per day (Day shift + Night shift)

print("Loading fact_crew_hours...")
watermark = get_watermark("fact_crew_hours")

spark.sql(f"""
    INSERT INTO workspace.bronze.fact_crew_hours
    SELECT
        -- date columns
        CAST(DATE_FORMAT(TO_TIMESTAMP(s.recorded_at, '{TS_FMT}'), 'yyyyMMdd') AS INT)
                                                        AS date_id,
        TO_TIMESTAMP(s.recorded_at, '{TS_FMT}')         AS recorded_at,

        -- surrogate keys
        COALESCE(r.rig_id_sk,    -1)                    AS rig_id_sk,
        COALESCE(rg.region_id_sk,-1)                    AS region_id_sk,
        COALESCE(c.crew_id_sk,   -1)                    AS crew_id_sk,
        COALESCE(sh.shift_id_sk, -1)                    AS shift_id_sk,

        -- natural keys (traceability)
        s.rig_id,
        s.crew_id,

        -- measures
        s.hours_worked,
        s.overtime_hours,
        s.is_present,
        s.npt_attributed,

        -- audit
        current_timestamp()                             AS loaded_at

    FROM workspace.staging.stg_crew s

    LEFT JOIN workspace.bronze.dim_rig r
        ON  s.rig_id     = r.rig_id
        AND r.is_current = true

    LEFT JOIN workspace.bronze.dim_region rg
        ON  r.region_id_sk = rg.region_id_sk
        AND rg.is_current  = true

    LEFT JOIN workspace.bronze.dim_crew_member c
        ON  s.crew_id    = c.crew_id
        AND c.is_current = true

    LEFT JOIN workspace.bronze.dim_shift sh
        ON  s.shift       = sh.shift_name
        AND sh.is_current = true

    WHERE CAST(DATE_FORMAT(TO_TIMESTAMP(s.recorded_at, '{TS_FMT}'), 'yyyyMMdd') AS INT) > {watermark}
""")

count = spark.sql("SELECT COUNT(*) as n FROM workspace.bronze.fact_crew_hours").collect()[0]["n"]
if count == 0:
    raise Exception("fact_crew_hours has 0 rows after load — aborting")
print(f"✓ workspace.bronze.fact_crew_hours: {count:,} rows")

Loading fact_crew_hours...
  → fact_crew_hours watermark: 0
✓ workspace.bronze.fact_crew_hours: 35,240 rows


In [0]:
# CELL 8 — FACT_INCIDENT_LOG
# Grain : incident_id
# Note  : equipment_type is nullable — NULL joins to -1 via COALESCE

print("Loading fact_incident_log...")
watermark = get_watermark("fact_incident_log")

spark.sql(f"""
    INSERT INTO workspace.bronze.fact_incident_log
    SELECT
        -- natural business ID
        s.incident_id,

        -- date columns
        CAST(DATE_FORMAT(TO_TIMESTAMP(s.recorded_at, '{TS_FMT}'), 'yyyyMMdd') AS INT)
                                                        AS date_id,
        TO_TIMESTAMP(s.recorded_at, '{TS_FMT}')         AS recorded_at,

        -- surrogate keys
        COALESCE(r.rig_id_sk,          -1)              AS rig_id_sk,
        COALESCE(rg.region_id_sk,      -1)              AS region_id_sk,
        COALESCE(c.crew_id_sk,         -1)              AS crew_id_sk,
        COALESCE(it.incident_type_id_sk,-1)             AS incident_type_id_sk,
        COALESCE(e.equipment_id_sk,    -1)              AS equipment_id_sk,
        COALESCE(st.status_id_sk,      -1)              AS status_id_sk,

        -- natural keys (traceability)
        s.rig_id,

        -- measures
        s.downtime_caused_hrs,
        s.is_recordable,

        -- audit
        current_timestamp()                             AS loaded_at

    FROM workspace.staging.stg_incidents s

    LEFT JOIN workspace.bronze.dim_rig r
        ON  s.rig_id     = r.rig_id
        AND r.is_current = true

    LEFT JOIN workspace.bronze.dim_region rg
        ON  r.region_id_sk = rg.region_id_sk
        AND rg.is_current  = true

    LEFT JOIN workspace.bronze.dim_crew_member c
        ON  s.crew_id    = c.crew_id
        AND c.is_current = true

    LEFT JOIN workspace.bronze.dim_incident_type it
        ON  s.incident_type = it.incident_type
        AND it.is_current   = true

    LEFT JOIN workspace.bronze.dim_equipment e
        ON  s.equipment_type = e.equipment_type
        AND e.is_current     = true

    LEFT JOIN workspace.bronze.dim_status st
        ON  s.incident_status = st.status_name
        AND st.is_current     = true

    WHERE CAST(DATE_FORMAT(TO_TIMESTAMP(s.recorded_at, '{TS_FMT}'), 'yyyyMMdd') AS INT) > {watermark}
""")

count = spark.sql("SELECT COUNT(*) as n FROM workspace.bronze.fact_incident_log").collect()[0]["n"]
if count == 0:
    raise Exception("fact_incident_log has 0 rows after load — aborting")
print(f"✓ workspace.bronze.fact_incident_log: {count:,} rows")

Loading fact_incident_log...
  → fact_incident_log watermark: 0
✓ workspace.bronze.fact_incident_log: 1,465 rows


In [0]:
# CELL 9 — RAW ARCHIVE LOAD
# Exact mirror of staging — append only, grows forever
# Watermark on recorded_at date_id — only new dates appended
# Never truncated — this is the permanent audit trail

print("Loading raw archive tables...")

# Create raw tables if not exist (schema mirrors staging exactly)
for stg, raw in [
    ("stg_rig_readings",     "raw_rig_readings"),
    ("stg_equipment_events", "raw_equipment_events"),
    ("stg_maintenance",      "raw_maintenance"),
    ("stg_crew",             "raw_crew"),
    ("stg_incidents",        "raw_incidents"),
]:
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS workspace.raw.{raw}
        USING DELTA
        AS SELECT *, current_timestamp() AS raw_loaded_at
        FROM workspace.staging.{stg}
        WHERE 1=0
    """)

print("✓ Raw tables ready")

# Append helper — only inserts dates not yet in raw
def append_raw(stg_table: str, raw_table: str):
    wm_row = spark.sql(f"""
        SELECT MAX(
            CAST(DATE_FORMAT(
                TO_TIMESTAMP(recorded_at, '{TS_FMT}'),
                'yyyyMMdd'
            ) AS INT)
        ) as max_id
        FROM workspace.raw.{raw_table}
    """).collect()[0]["max_id"]
    wm = wm_row if wm_row else 0

    spark.sql(f"""
        INSERT INTO workspace.raw.{raw_table}
        SELECT *, current_timestamp() AS raw_loaded_at
        FROM workspace.staging.{stg_table}
        WHERE CAST(DATE_FORMAT(
            TO_TIMESTAMP(recorded_at, '{TS_FMT}'),
            'yyyyMMdd'
        ) AS INT) > {wm}
    """)

    count = spark.sql(
        f"SELECT COUNT(*) as n FROM workspace.raw.{raw_table}"
    ).collect()[0]["n"]
    print(f"  ✓ workspace.raw.{raw_table}: {count:,} rows total")

append_raw("stg_rig_readings",     "raw_rig_readings")
append_raw("stg_equipment_events", "raw_equipment_events")
append_raw("stg_maintenance",      "raw_maintenance")
append_raw("stg_crew",             "raw_crew")
append_raw("stg_incidents",        "raw_incidents")

print("\n✓ Raw archive load complete")

Loading raw archive tables...
✓ Raw tables ready
  ✓ workspace.raw.raw_rig_readings: 17,620 rows total
  ✓ workspace.raw.raw_equipment_events: 88,100 rows total
  ✓ workspace.raw.raw_maintenance: 8,672 rows total
  ✓ workspace.raw.raw_crew: 35,240 rows total
  ✓ workspace.raw.raw_incidents: 1,465 rows total

✓ Raw archive load complete


In [0]:
# CELL 10 — VERIFICATION
# Row counts, column counts, natural key check, -1 SK check
# Raises exception if any table is empty

bronze_facts = [
    "fact_rig_daily_ops",
    "fact_equipment_events",
    "fact_maintenance_log",
    "fact_crew_hours",
    "fact_incident_log",
]
raw_tables = [
    "raw_rig_readings",
    "raw_equipment_events",
    "raw_maintenance",
    "raw_crew",
    "raw_incidents",
]

print("=" * 70)
print("BRONZE FACT TABLES")
print("=" * 70)
print(f"{'Table':<35} {'Rows':>10}  {'Cols':>5}  {'rig_id':>8}  {'loaded_at':>10}")
print("-" * 70)

bronze_total = 0
issues       = []

for t in bronze_facts:
    df           = spark.table(f"workspace.bronze.{t}")
    count        = df.count()
    cols         = len(df.columns)
    has_rig_id   = "rig_id" in df.columns
    has_loaded   = "loaded_at" in df.columns
    bronze_total += count

    if count == 0:
        issues.append(f"  ⚠ workspace.bronze.{t} is empty")

    # Verify -1 SKs exist (means unknown member is working)
    sk_cols = [c for c in df.columns if c.endswith("_id_sk")]
    neg1_ok = True
    for sk in sk_cols:
        neg1_count = df.filter(F.col(sk) == -1).count()
        if neg1_count > 0:
            pass  # expected — some rows resolved to unknown

    print(f"workspace.bronze.{t:<19} {count:>10,}  {cols:>5}  "
          f"{'✓' if has_rig_id else '✗':>8}  "
          f"{'✓' if has_loaded else '✗':>10}")

print("-" * 70)
print(f"{'Bronze total':<35} {bronze_total:>10,}")

print("\n" + "=" * 70)
print("RAW ARCHIVE TABLES")
print("=" * 70)

raw_total = 0
for t in raw_tables:
    count      = spark.sql(f"SELECT COUNT(*) as n FROM workspace.raw.{t}").collect()[0]["n"]
    raw_total += count
    print(f"  workspace.raw.{t:<28} {count:>10,} rows")

print(f"\n  Raw total: {raw_total:,} rows")

if issues:
    print("\nISSUES FOUND:")
    for i in issues:
        print(i)
    raise Exception(
        "Bronze verification failed — "
        "do not proceed to Notebook_1_SQL"
    )
else:
    print("\n✓ All bronze facts and raw tables verified")
    print("✓ Notebook 0c complete — safe to proceed to Notebook_1_SQL")

BRONZE FACT TABLES
Table                                     Rows   Cols    rig_id   loaded_at
----------------------------------------------------------------------
workspace.bronze.fact_rig_daily_ops      17,620     18         ✓           ✓
workspace.bronze.fact_equipment_events     88,100     13         ✓           ✓
workspace.bronze.fact_maintenance_log      8,672     13         ✓           ✓
workspace.bronze.fact_crew_hours         35,240     13         ✓           ✓
workspace.bronze.fact_incident_log        1,465     13         ✓           ✓
----------------------------------------------------------------------
Bronze total                           151,097

RAW ARCHIVE TABLES
  workspace.raw.raw_rig_readings                 17,620 rows
  workspace.raw.raw_equipment_events             88,100 rows
  workspace.raw.raw_maintenance                   8,672 rows
  workspace.raw.raw_crew                         35,240 rows
  workspace.raw.raw_incidents                     1,465 rows

  